In [ ]:
from droid.robot_env import RobotEnv
import sys
try:
    sys.path.append("/home/abhagejji/eai/jepa-new")
    from scripts.robot_wm.gripper_interface import GripperInterface
except ImportError:
    print("Unable to import gripper interface. Ensure the path is correct.")

In [ ]:
# Initialize the Panda environment. Using joint velocity action space and gripper position action space is very important.
env = RobotEnv(action_space="joint_velocity", gripper_action_space="position")
print("Created the droid env!")
# Gripper Interface
gripper_interface = GripperInterface(ip_address="localhost", port=50052)

def get_gripper_position(gripper_interface = gripper_interface):
    return 1 - (gripper_interface.get_state().width / gripper_interface.metadata.max_width)

In [ ]:
# Initialize the Panda environment. Using joint velocity action space and gripper position action space is very important.
env = RobotEnv(action_space="cartesian_position", gripper_action_space="position")
print("Created the droid env!")
# Gripper Interface
gripper_interface = GripperInterface(ip_address="localhost", port=50052)

def get_gripper_position(gripper_interface = gripper_interface):
    return 1 - (gripper_interface.get_state().width / gripper_interface.metadata.max_width)

In [ ]:

# gripper range is 0 to 0.085
target_gripper_width = 0.085 # and 0 is closed
gripper_interface.goto(width=target_gripper_width, speed=0.5, force=1)
gripper_interface.get_state()

In [ ]:
env.go_home()

In [ ]:
kp = env.get_kp_gains()
print("KP Gains:", kp)
kd = env.get_kd_gains()
print("KD Gains:", kd)
kx = env.get_kx_gains()
print("KX Gains:", kx)
kxd = env.get_kxd_gains()
print("KXD Gains:", kxd)

In [ ]:
# remote franka controller
import Pyro5.api
import numpy as np
from matplotlib import pyplot as plt
import cv2
import time

# DROID data collection frequency -- we slow down execution to match this frequency
DROID_CONTROL_FREQUENCY = 15

frame_buffer = []

polymetis_actions = []


# reset to home position
env.go_home()
gripper_interface.goto(width=0.085, speed=0.5, force=1)
max_gripper_width = gripper_interface.metadata.max_width


# pi0_fast_actions = []
# pi0_actions = []



@Pyro5.api.expose
class ControllerInterface:
    def __init__(self):
        self.step_count = 0

    def step(self, data_dict):  # data_dict: {'action': [...], 'step': int}
        action_chunk = data_dict["action"]
        execution_num = len(action_chunk) - 2
        for i in range(execution_num):
            start_time = time.time()
            action = action_chunk[i]
            action = np.array(action) # action is joint velocity
            # polymetis_actions.append(action)
            # pi0_actions.append(action)
            # pi0_fast_actions.append(action)
            # joint_action = action[:-1]  # all but the last element are joint actions, joint velocoty
            gripper_action = action[-1]  # last element is gripper width
            gripper_action = float(np.clip(gripper_action, 0, 1))

            # delta_joint = joint_action * robot_interface._control_interval # convert to joint position change
            # delta_joint = joint_action * 0.1 # convert to joint position change
            # binarize gripper action
            if gripper_action < 0.5:
                target_gripper_width = 1.0  # open the gripper
            elif gripper_action >= 0.5:
                target_gripper_width = 0.0
            # else:
            #     target_gripper_width = 1.0

            target_gripper_width = max_gripper_width * (target_gripper_width)
            action = np.clip(action, -1, 1)
            action_dict = env.step(action)
            gripper_interface.goto(width=target_gripper_width, speed=0.05, force=0.1, blocking=False)
            elapsed_time = time.time() - start_time
            if elapsed_time < 1 / DROID_CONTROL_FREQUENCY:
                time.sleep(1 / DROID_CONTROL_FREQUENCY - elapsed_time)

        print(f"control Step {self.step_count} | Received action: {action_chunk[0]}")


        start = time.time() 
        # get robot and gripper state
        # robot_state = list(robot_interface._state_buffer[-1].q)
        robot_state = env.get_joint_positions()
        # gripper_state = float(gripper_interface.get_state().width)
        
        # gripper_state = 1 - (gripper_interface.get_state().width / max_gripper_width)
        gripper_state = get_gripper_position(gripper_interface = gripper_interface)
        # print(f" Gripper state: {gripper_state}")
        # print('---------------------------------')
        self.step_count += 1
        return {
            "robot_state": robot_state,
            "gripper_state": gripper_state,
            "step": self.step_count
        }

# Pyro5 server
daemon = Pyro5.api.Daemon(host="localhost")
ns = Pyro5.api.locate_ns()  # Locate the name server
uri = daemon.register(ControllerInterface)
ns.register("pi0_controller", uri)  # Register the object with the name server
print("Controller server running at:")
print(uri)
daemon.requestLoop()




In [ ]:
# remote franka controller
import Pyro5.api
import numpy as np
from matplotlib import pyplot as plt
import cv2
import time
from scipy.spatial.transform import Rotation as R

def get_gripper_position(gripper_interface = gripper_interface):
    return 1 - (gripper_interface.get_state().width / gripper_interface.metadata.max_width)


DROID_CONTROL_FREQUENCY_HZ = 15  # Hz
execute_n_actions = 8

frame_buffer = []

# reset to home position
env.go_home()
gripper_interface.goto(width=0.085, speed=0.5, force=1)





@Pyro5.api.expose
class ControllerInterface:
    def __init__(self):
        self.step_count = 0

    def step(self, data_dict):  # data_dict: {'action': [...], 'step': int}
        action_chunk = data_dict["action"]
        for i in range(execute_n_actions):
            start_time = time.time()
            action = action_chunk[i]
            action = np.array(action) # delta eef
            delta_eef_action = action *0.05
            current_pose = env.get_ee_pose() + [0.0]
            state = np.array(current_pose)
            state_action = state + delta_eef_action


            gripper_state_action = action[-1]  # last element is gripper width
            print(f"Gripper state action: {gripper_state_action}")
            target_gripper_width = 1.0 - gripper_state_action  # convert to width


            action_dict = env.step(state_action)
            # gripper action
            gripper_interface.goto(width=target_gripper_width, speed=0.05, force=0.1, blocking=False)

            elapsed_time = time.time() - start_time
            if elapsed_time < 1 / DROID_CONTROL_FREQUENCY_HZ:
                time.sleep(1 / DROID_CONTROL_FREQUENCY_HZ - elapsed_time)

        print(f"control Step {self.step_count} | Received action: {action}")


        # get robot and gripper state
        current_pose = env.get_ee_pose()
        current_pos = np.array(current_pose[:3]).reshape(1, -1)
        euler_angles = np.array(current_pose[3:]).reshape(1, -1)
        # current_qpos = list(robot_interface.last_q) + [0.0, 0.0]
        # current_qpos = np.array(current_qpos)

        # current_pos, euler_angles = get_eef_pose_from_qpos(kin_helper, current_qpos)
        # current_pos = np.array(current_pos).reshape(1, -1)
        # euler_angles = np.array(euler_angles).reshape(1, -1)
        # gripper_state = np.array(gripper_interface.get_state().width).reshape(1, -1)
        gripper_state = np.array(get_gripper_position(gripper_interface = gripper_interface)).reshape(1, -1)

        self.step_count += 1
        return {
            "robot_pos": current_pos.tolist(),
            "robot_rot": euler_angles.tolist(),
            "gripper_state": gripper_state.tolist(),
            "step": self.step_count
        }

# Pyro5 server
daemon = Pyro5.api.Daemon(host="localhost")
ns = Pyro5.api.locate_ns()  # Locate the name server
uri = daemon.register(ControllerInterface)
ns.register("gr00t_controller", uri)  # Register the object with the name server
print("Controller server running at:")
print(uri)
daemon.requestLoop()

